In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
df.head()

df.info()
df.describe()
df.isnull().sum()

# 1. Family size (SibSp = siblings/spouses, Parch = parents/children)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# 2. Is the person alone?
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 3. Extract title from name (Mr, Mrs, Miss, Master...)
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
print(df['Title'].value_counts())

# 4. Group rare titles
df['Title'] = df['Title'].replace(['Lady','Countess','Capt','Col','Don',
                                    'Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
df['Title'] = df['Title'].replace({'Mlle':'Miss', 'Ms':'Miss', 'Mme':'Mrs'})

# 5. Age groups (binning)
df['AgeGroup'] = pd.cut(df['Age'], bins=[0,12,18,35,60,100],
                         labels=['Child','Teen','Young Adult','Adult','Senior'])

# 6. Fare per person
df['FarePerPerson'] = df['Fare'] / df['FamilySize']


# Fill missing Age with median of same Title group
df['Age'] = df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Sex_encoded'] = le.fit_transform(df['Sex'])          # male=1, female=0
df['Title_encoded'] = le.fit_transform(df['Title'])
df['Embarked'] = df['Embarked'].fillna('S')
df['Embarked_encoded'] = le.fit_transform(df['Embarked'])


from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

features = ['Pclass','Sex_encoded','Age','FamilySize','IsAlone',
            'FarePerPerson','Title_encoded','Embarked_encoded']

X = df[features].fillna(0)
y = df['Survived']

model = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"Accuracy with engineered features: {scores.mean():.3f}")

raw_features = ['Pclass', 'Sex_encoded', 'Age', 'Fare', 'SibSp', 'Parch']
X_raw = df[raw_features].fillna(0)

scores_raw = cross_val_score(model, X_raw, y, cv=5, scoring='accuracy')
print(f"Accuracy WITHOUT feature engineering: {scores_raw.mean():.3f}")
print(f"Accuracy WITH feature engineering:    {scores.mean():.3f}")